# Classical Axis 2 Comparison

This notebook has the code about the **classical Axis 2 comparison**. It includes the minimum upstream steps required to run Section 9 independently: imports, data loading, preprocessing, fixed CV/test splits, evaluation helpers, the baseline, A2a-A2e classical models, and the final summary plots.


In [ ]:
# SECTION 1 — Imports and global setup
import os
import re
import json
import math
import random
import warnings
import urllib.request
from pathlib import Path
from collections import defaultdict
from functools import reduce

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.sparse import hstack
from scipy.special import softmax as sp_softmax

from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import VotingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

warnings.filterwarnings("default")

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 10

save_dir = "Visualization_axis2_standalone"
os.makedirs(save_dir, exist_ok=True)

try:
    display
except NameError:
    def display(x):
        print(x)

print("Core libraries imported successfully.")
print(f"Random seed set to {RANDOM_SEED}")
print(f"Figures will be saved to: {save_dir}")


## Section 2 — Data Loading

The notebook first looks for `ori_pqal.json` in common local locations, including the original coursework folders. If it cannot find the file, it attempts to download the public PubMedQA labelled dataset from the official GitHub repository.


In [ ]:
# SECTION 2 — Data loading
candidate_paths = [
    Path("/content/data/ori_pqal.json"),
    Path("data/ori_pqal.json"),
    Path("./data/ori_pqal.json"),
    Path("/mnt/data/ori_pqal.json"),
    Path("/Users/xuezhengjie/Desktop/ori_pqal.json"),
    Path("/Users/xuezhengjie/Desktop/data/ori_pqal.json"),
    Path("/Users/xuezhengjie/Desktop/小组作业/reference files/ori_pqal.json"),
    Path("/Users/xuezhengjie/Desktop/ai-text-analytics-coursework-team16-task1-main/data/ori_pqal.json"),
    Path("/Users/xuezhengjie/Desktop/ai-text-analytics-coursework-team16-task1-main/code/data/ori_pqal.json"),
]

DATA_PATH = next((p for p in candidate_paths if p.exists()), None)

if DATA_PATH is None:
    os.makedirs("data", exist_ok=True)
    DATA_PATH = Path("data/ori_pqal.json")
    url = "https://raw.githubusercontent.com/pubmedqa/pubmedqa/master/data/ori_pqal.json"
    print("ori_pqal.json was not found locally. Downloading PubMedQA PQA-L from:")
    print(url)
    urllib.request.urlretrieve(url, DATA_PATH)

print(f"Using data file: {DATA_PATH}")

with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Total records: {len(data)}")

rows = []
for pmid, info in data.items():
    contexts = info.get("CONTEXTS", [])
    rows.append({
        "pmid": pmid,
        "question": info.get("QUESTION", ""),
        "context_list": contexts,
        "context": " ".join(contexts) if isinstance(contexts, list) else str(contexts),
        "long_answer": info.get("LONG_ANSWER", ""),
        "label": str(info.get("final_decision", "")).lower().strip(),
    })

df = pd.DataFrame(rows)

print(f"DataFrame shape: {df.shape}")
print("Label distribution:")
print(df["label"].value_counts())

if df.empty or "label" not in df.columns:
    raise ValueError("Dataset did not load correctly. Please check DATA_PATH.")


## Section 3 — Preprocessing and Text Fields

This reproduces the cleaned text fields used by the full notebook. Axis 2 uses the `q_ctx` representation so the classifier/training design is what changes.


In [ ]:
# SECTION 3 — Text preprocessing and feature engineering
df_model = df.copy()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s\-\?\.,:;/%()]", " ", text)
    return text.strip()

df_model["question_clean"] = df_model["question"].apply(clean_text)
df_model["context_clean"] = df_model["context"].apply(clean_text)
df_model["long_answer_clean"] = df_model["long_answer"].apply(clean_text)

label2id = {"no": 0, "maybe": 1, "yes": 2}
id2label = {0: "no", 1: "maybe", 2: "yes"}
df_model["label_id"] = df_model["label"].map(label2id)

assert df_model["label_id"].isna().sum() == 0, "Unmapped labels found. Expected labels: no/maybe/yes."

for frame in [df_model]:
    frame["question_len"] = frame["question_clean"].apply(lambda x: len(x.split()))
    frame["context_len"] = frame["context_clean"].apply(lambda x: len(x.split()))
    frame["long_answer_len"] = frame["long_answer_clean"].apply(lambda x: len(x.split()))
    frame["num_contexts"] = frame["context_list"].apply(lambda x: len(x) if isinstance(x, list) else 0)

print("Preprocessing complete")
print(f"Shape: {df_model.shape}")
print(df_model["label"].value_counts())


## Section 4 — Data Splits

This recreates the original 500-sample CV set and 500-sample held-out test set, then assigns 10 stratified folds inside the CV set.


In [ ]:
# SECTION 4 — Data splits
random.seed(RANDOM_SEED)

def split_dataset(dataset_pmids, labels, fold):
    add = lambda x: reduce(lambda a, b: a + b, x)

    label2pmids = defaultdict(list)
    for pmid, lab in zip(dataset_pmids, labels):
        label2pmids[lab].append(pmid)

    label2splits = {}
    for lab, pmids in label2pmids.items():
        pmids = pmids.copy()
        random.shuffle(pmids)
        num_all = len(pmids)
        num_split = math.ceil(num_all / fold)
        splits = []
        for i in range(fold):
            if i == fold - 1:
                splits.append(pmids[i * num_split:])
            else:
                splits.append(pmids[i * num_split: (i + 1) * num_split])
        label2splits[lab] = splits

    output = []
    for i in range(fold):
        fold_pmids = add([label2splits[lab][i] for lab in sorted(label2splits.keys())])
        output.append(fold_pmids)

    if len(output[-1]) != len(output[0]):
        for i in range(fold - 1):
            while len(output[i]) > len(output[-1]):
                picked = random.choice(output[i])
                output[-1].append(picked)
                output[i].remove(picked)

    return output

all_pmids = df_model["pmid"].tolist()
all_labels = df_model["label"].tolist()

two_halves = split_dataset(all_pmids, all_labels, 2)
cv_pmids = set(two_halves[0])
test_pmids = set(two_halves[1])

cv_df = df_model[df_model["pmid"].isin(cv_pmids)].copy()
test_df = df_model[df_model["pmid"].isin(test_pmids)].copy()

print(f"CV set:   {len(cv_df)} samples")
print(f"Test set: {len(test_df)} samples")
print("\nCV label distribution:")
print(cv_df["label"].value_counts())
print("\nTest label distribution:")
print(test_df["label"].value_counts())

assert len(cv_df) == 500, f"CV set should be 500, got {len(cv_df)}"
assert len(test_df) == 500, f"Test set should be 500, got {len(test_df)}"
assert set(cv_df["pmid"]).isdisjoint(set(test_df["pmid"])), "CV/Test overlap detected!"

cv_df = cv_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_SEED)

cv_df["fold"] = -1
for fold_idx, (_, val_idx) in enumerate(skf.split(cv_df, cv_df["label"])):
    cv_df.loc[val_idx, "fold"] = fold_idx

q_col = "question_clean"
c_col = "context_clean"
a_col = "long_answer_clean"

for frame in [cv_df, test_df]:
    frame["q_only"] = frame[q_col].fillna("")
    frame["q_ctx"] = frame[q_col].fillna("") + " [SEP] " + frame[c_col].fillna("")
    frame["q_ctx_ans"] = frame[q_col].fillna("") + " [SEP] " + frame[c_col].fillna("") + " [SEP] " + frame[a_col].fillna("")

y_test = test_df["label_id"].values
pmid_test = test_df["pmid"].values
X_test_qctx = test_df["q_ctx"].tolist()

y_cv = cv_df["label_id"].values
X_cv_qctx = cv_df["q_ctx"].tolist()

print("\n10-fold CV split sizes:")
for i in range(10):
    fold_df = cv_df[cv_df["fold"] == i]
    print(f"  Fold {i}: {len(fold_df)} samples — {fold_df['label'].value_counts().to_dict()}")


## Section 5 — Evaluation Helpers

These helpers keep TF-IDF fitting inside each fold to avoid leakage. The word+character TF-IDF feature builder is used by all classical Axis 2 variants.


In [ ]:
# SECTION 5 — Evaluation helpers
def evaluate_10fold_cv(make_pipeline_fn, cv_df, text_col, test_texts, y_test, n_folds=10):
    fold_accs = []
    fold_f1s = []

    for fold_idx in range(n_folds):
        val_mask = cv_df["fold"] == fold_idx
        train_mask = ~val_mask

        X_tr = cv_df.loc[train_mask, text_col].tolist()
        y_tr = cv_df.loc[train_mask, "label_id"].values
        X_val = cv_df.loc[val_mask, text_col].tolist()
        y_val = cv_df.loc[val_mask, "label_id"].values

        pipe = make_pipeline_fn()
        pipe.fit(X_tr, y_tr)
        val_preds = pipe.predict(X_val)

        fold_accs.append(accuracy_score(y_val, val_preds))
        fold_f1s.append(f1_score(y_val, val_preds, average="macro", zero_division=0))

    final_pipe = make_pipeline_fn()
    final_pipe.fit(cv_df[text_col].tolist(), cv_df["label_id"].values)
    test_preds = final_pipe.predict(test_texts)

    return {
        "cv_acc_mean": np.mean(fold_accs),
        "cv_acc_std": np.std(fold_accs),
        "cv_f1_mean": np.mean(fold_f1s),
        "cv_f1_std": np.std(fold_f1s),
        "test_acc": accuracy_score(y_test, test_preds),
        "test_f1": f1_score(y_test, test_preds, average="macro", zero_division=0),
        "preds": test_preds,
        "final_pipeline": final_pipe,
    }

def build_word_char_features(train_texts, other_texts, max_word=40000, max_char=30000):
    word_vec = TfidfVectorizer(
        lowercase=True, stop_words="english",
        ngram_range=(1, 2), min_df=2,
        sublinear_tf=True, max_features=max_word
    )
    char_vec = TfidfVectorizer(
        lowercase=True, analyzer="char_wb",
        ngram_range=(3, 5), min_df=2,
        sublinear_tf=True, max_features=max_char
    )
    X_train_word = word_vec.fit_transform(train_texts)
    X_train_char = char_vec.fit_transform(train_texts)
    X_train = hstack([X_train_word, X_train_char]).tocsr()

    outputs = []
    for texts in other_texts:
        X_word = word_vec.transform(texts)
        X_char = char_vec.transform(texts)
        outputs.append(hstack([X_word, X_char]).tocsr())

    return X_train, outputs, word_vec, char_vec

def maybe_threshold_predict(probs, maybe_threshold=0.40, maybe_class_id=1):
    non_maybe_best = np.where(probs[:, 0] >= probs[:, 2], 0, 2)
    preds = np.where(probs[:, maybe_class_id] >= maybe_threshold, maybe_class_id, non_maybe_best)
    return preds

print("Evaluation helpers ready.")


## Section 6 — Baseline TF-IDF + Logistic Regression

This is the original baseline needed for comparison with the Axis 2 classical models.


In [ ]:
# SECTION 6 — Baseline TF-IDF + Logistic Regression
def make_baseline_pipe():
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True, stop_words="english",
            ngram_range=(1, 2), max_features=30000
        )),
        ("clf", LogisticRegression(
            max_iter=2000, class_weight="balanced",
            random_state=RANDOM_SEED, C=1.0
        ))
    ])

baseline_res = evaluate_10fold_cv(make_baseline_pipe, cv_df, "q_ctx", X_test_qctx, y_test)

print("=" * 60)
print("BASELINE: TF-IDF (1,2)-grams + Logistic Regression")
print("=" * 60)
print(f"10-Fold CV Accuracy : {baseline_res['cv_acc_mean']:.4f} ± {baseline_res['cv_acc_std']:.4f}")
print(f"10-Fold CV Macro-F1 : {baseline_res['cv_f1_mean']:.4f} ± {baseline_res['cv_f1_std']:.4f}")
print(f"Test Accuracy       : {baseline_res['test_acc']:.4f}")
print(f"Test Macro-F1       : {baseline_res['test_f1']:.4f}")
print(f"\n{classification_report(y_test, baseline_res['preds'], target_names=['no','maybe','yes'], zero_division=0)}")

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, baseline_res["preds"],
    display_labels=["no", "maybe", "yes"], ax=ax, cmap="Blues"
)
ax.set_title("Baseline Confusion Matrix (Test Set, n=500)")
plt.tight_layout()
plt.savefig(os.path.join(save_dir, "Baseline_Confusion_Matrix.png"), dpi=300, bbox_inches="tight")
plt.show()

all_results = {
    "Baseline: TF-IDF+LR": {
        "CV Macro-F1": f"{baseline_res['cv_f1_mean']:.4f} ± {baseline_res['cv_f1_std']:.4f}",
        "Test Acc": baseline_res["test_acc"],
        "Test Macro-F1": baseline_res["test_f1"],
    }
}
all_preds = {"Baseline": baseline_res["preds"]}


## Section 7 — Axis 2 Block 1: Tuned Classical Classifiers

A2a-A2c compare three classical classifiers using the same word+character TF-IDF representation: tuned Logistic Regression, calibrated Linear SVM with maybe-threshold tuning, and Multinomial Naive Bayes.


In [ ]:
# SECTION 9 — AXIS 2: CLASSIFIER & TRAINING DESIGN
# BLOCK 1 — PART A (A2a, A2b, A2c)

def evaluate_axis2_10fold(cv_df, test_texts, y_test, text_col="q_ctx"):
    axis2_results = {}
    axis2_preds_dict = {}

    # Shared final features for all word+char models
    X_cv_wc, [X_test_wc], word_vec_final, char_vec_final = build_word_char_features(
        cv_df[text_col].tolist(), [test_texts]
    )

    # A2a: Tuned LR (word+char TF-IDF)
    print("A2a — Tuned Logistic Regression (word+char TF-IDF)")
    fold_f1s_lr = []

    for fold_idx in range(10):
        val_mask = cv_df["fold"] == fold_idx
        train_mask = ~val_mask

        X_tr_texts = cv_df.loc[train_mask, text_col].tolist()
        y_tr = cv_df.loc[train_mask, "label_id"].values
        X_va_texts = cv_df.loc[val_mask, text_col].tolist()
        y_va = cv_df.loc[val_mask, "label_id"].values

        X_tr_wc, [X_va_wc], _, _ = build_word_char_features(X_tr_texts, [X_va_texts])

        inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
        lr_grid = GridSearchCV(
            LogisticRegression(
                max_iter=4000,
                class_weight="balanced",
                solver="liblinear",
                random_state=RANDOM_SEED
            ),
            param_grid={"C": [0.05, 0.1, 0.5, 1.0, 2.0, 5.0]},
            cv=inner_cv,
            scoring="f1_macro",
            refit=True,
            n_jobs=-1
        )
        lr_grid.fit(X_tr_wc, y_tr)
        val_preds = lr_grid.predict(X_va_wc)
        fold_f1s_lr.append(f1_score(y_va, val_preds, average="macro"))

    lr_final = GridSearchCV(
        LogisticRegression(
            max_iter=4000,
            class_weight="balanced",
            solver="liblinear",
            random_state=RANDOM_SEED
        ),
        param_grid={"C": [0.05, 0.1, 0.5, 1.0, 2.0, 5.0]},
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED),
        scoring="f1_macro",
        refit=True,
        n_jobs=-1
    )
    lr_final.fit(X_cv_wc, y_cv)
    pred_lr = lr_final.predict(X_test_wc)

    print(f"  Best C              : {lr_final.best_params_['C']}")
    print(f"  10-Fold CV Macro-F1 : {np.mean(fold_f1s_lr):.4f} ± {np.std(fold_f1s_lr):.4f}")
    print(f"  Test Acc            : {accuracy_score(y_test, pred_lr):.4f}")
    print(f"  Test Macro-F1       : {f1_score(y_test, pred_lr, average='macro'):.4f}")
    print(classification_report(y_test, pred_lr, target_names=["no", "maybe", "yes"], zero_division=0))

    axis2_results["A2a: LR word+char"] = {
        "CV Macro-F1": f"{np.mean(fold_f1s_lr):.4f} ± {np.std(fold_f1s_lr):.4f}",
        "Test Acc": accuracy_score(y_test, pred_lr),
        "Test Macro-F1": f1_score(y_test, pred_lr, average="macro")
    }
    axis2_preds_dict["A2a_LR_wordchar"] = pred_lr

    # A2b: Calibrated Linear SVM + maybe-threshold tuning 
    print("\nA2b — Calibrated Linear SVM + maybe-threshold tuning")
    fold_f1s_svm = []

    for fold_idx in range(10):
        val_mask = cv_df["fold"] == fold_idx
        train_mask = ~val_mask

        outer_train_df = cv_df.loc[train_mask].reset_index(drop=True)
        outer_val_df = cv_df.loc[val_mask].reset_index(drop=True)

        X_outer_train = outer_train_df[text_col].tolist()
        y_outer_train = outer_train_df["label_id"].values
        X_outer_val = outer_val_df[text_col].tolist()
        y_outer_val = outer_val_df["label_id"].values

        inner_splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
        inner_train_idx, inner_dev_idx = next(inner_splitter.split(X_outer_train, y_outer_train))

        X_inner_train = [X_outer_train[i] for i in inner_train_idx]
        y_inner_train = y_outer_train[inner_train_idx]
        X_inner_dev = [X_outer_train[i] for i in inner_dev_idx]
        y_inner_dev = y_outer_train[inner_dev_idx]

        X_inner_wc, [X_dev_wc, X_outer_val_wc], _, _ = build_word_char_features(
            X_inner_train, [X_inner_dev, X_outer_val]
        )

        svm_grid = GridSearchCV(
            LinearSVC(class_weight="balanced", random_state=RANDOM_SEED, max_iter=10000),
            param_grid={"C": [0.01, 0.05, 0.1, 0.5, 1.0, 2.0]},
            cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED),
            scoring="f1_macro",
            refit=True,
            n_jobs=-1
        )
        svm_grid.fit(X_inner_wc, y_inner_train)

        cal_svm_fold = CalibratedClassifierCV(
            svm_grid.best_estimator_,
            method="sigmoid",
            cv=5
        )
        cal_svm_fold.fit(X_inner_wc, y_inner_train)

        dev_probs = cal_svm_fold.predict_proba(X_dev_wc)
        best_thr_fold, best_dev_f1_fold = None, -1.0

        for thr in np.arange(0.25, 0.61, 0.025):
            dev_pred_thr = maybe_threshold_predict(dev_probs, maybe_threshold=thr, maybe_class_id=1)
            dev_f1 = f1_score(y_inner_dev, dev_pred_thr, average="macro")
            if dev_f1 > best_dev_f1_fold:
                best_dev_f1_fold = dev_f1
                best_thr_fold = float(thr)

        outer_val_probs = cal_svm_fold.predict_proba(X_outer_val_wc)
        outer_val_preds = maybe_threshold_predict(
            outer_val_probs,
            maybe_threshold=best_thr_fold,
            maybe_class_id=1
        )
        fold_f1s_svm.append(f1_score(y_outer_val, outer_val_preds, average="macro"))

    dev_fold_mask = cv_df["fold"] == 0
    train_no_dev_mask = ~dev_fold_mask

    X_tr_texts_svm = cv_df.loc[train_no_dev_mask, text_col].tolist()
    y_tr_svm = cv_df.loc[train_no_dev_mask, "label_id"].values
    X_dev_texts_svm = cv_df.loc[dev_fold_mask, text_col].tolist()
    y_dev_svm = cv_df.loc[dev_fold_mask, "label_id"].values

    X_tr_wc_s, [X_dev_wc_s, X_test_wc_s], _, _ = build_word_char_features(
        X_tr_texts_svm, [X_dev_texts_svm, test_texts]
    )

    svm_final = GridSearchCV(
        LinearSVC(class_weight="balanced", random_state=RANDOM_SEED, max_iter=10000),
        param_grid={"C": [0.01, 0.05, 0.1, 0.5, 1.0, 2.0]},
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED),
        scoring="f1_macro",
        refit=True,
        n_jobs=-1
    )
    svm_final.fit(X_tr_wc_s, y_tr_svm)

    cal_svm = CalibratedClassifierCV(
        svm_final.best_estimator_,
        method="sigmoid",
        cv=5
    )
    cal_svm.fit(X_tr_wc_s, y_tr_svm)

    dev_probs = cal_svm.predict_proba(X_dev_wc_s)
    best_thr, best_dev_f1 = None, -1.0
    for thr in np.arange(0.25, 0.61, 0.025):
        dev_pred_thr = maybe_threshold_predict(dev_probs, maybe_threshold=thr, maybe_class_id=1)
        dev_f1 = f1_score(y_dev_svm, dev_pred_thr, average="macro")
        if dev_f1 > best_dev_f1:
            best_dev_f1 = dev_f1
            best_thr = float(thr)

    test_probs_svm = cal_svm.predict_proba(X_test_wc_s)
    pred_svm_thr = maybe_threshold_predict(
        test_probs_svm,
        maybe_threshold=best_thr,
        maybe_class_id=1
    )

    print(f"  Best C              : {svm_final.best_params_['C']}")
    print(f"  Best dev threshold  : {best_thr:.3f}")
    print(f"  10-Fold CV Macro-F1 : {np.mean(fold_f1s_svm):.4f} ± {np.std(fold_f1s_svm):.4f}")
    print(f"  Test Acc            : {accuracy_score(y_test, pred_svm_thr):.4f}")
    print(f"  Test Macro-F1       : {f1_score(y_test, pred_svm_thr, average='macro'):.4f}")
    print(classification_report(y_test, pred_svm_thr, target_names=["no", "maybe", "yes"], zero_division=0))

    axis2_results["A2b: Cal-SVM word+char"] = {
        "CV Macro-F1": f"{np.mean(fold_f1s_svm):.4f} ± {np.std(fold_f1s_svm):.4f}",
        "Test Acc": accuracy_score(y_test, pred_svm_thr),
        "Test Macro-F1": f1_score(y_test, pred_svm_thr, average="macro")
    }
    axis2_preds_dict["A2b_CalSVM_wordchar"] = pred_svm_thr

    # A2c: Multinomial Naive Bayes 
    print("\nA2c — Multinomial Naive Bayes (word+char TF-IDF)")
    fold_f1s_nb = []

    for fold_idx in range(10):
        val_mask = cv_df["fold"] == fold_idx
        train_mask = ~val_mask

        X_tr_texts = cv_df.loc[train_mask, text_col].tolist()
        y_tr = cv_df.loc[train_mask, "label_id"].values
        X_va_texts = cv_df.loc[val_mask, text_col].tolist()
        y_va = cv_df.loc[val_mask, "label_id"].values

        X_tr_wc, [X_va_wc], _, _ = build_word_char_features(X_tr_texts, [X_va_texts])

        nb_grid = GridSearchCV(
            MultinomialNB(),
            param_grid={"alpha": [0.05, 0.1, 0.25, 0.5, 1.0, 2.0]},
            cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED),
            scoring="f1_macro",
            refit=True,
            n_jobs=-1
        )
        nb_grid.fit(X_tr_wc, y_tr)
        val_preds = nb_grid.predict(X_va_wc)
        fold_f1s_nb.append(f1_score(y_va, val_preds, average="macro"))

    nb_final = GridSearchCV(
        MultinomialNB(),
        param_grid={"alpha": [0.05, 0.1, 0.25, 0.5, 1.0, 2.0]},
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED),
        scoring="f1_macro",
        refit=True,
        n_jobs=-1
    )
    nb_final.fit(X_cv_wc, y_cv)
    pred_nb = nb_final.predict(X_test_wc)

    print(f"  Best alpha          : {nb_final.best_params_['alpha']}")
    print(f"  10-Fold CV Macro-F1 : {np.mean(fold_f1s_nb):.4f} ± {np.std(fold_f1s_nb):.4f}")
    print(f"  Test Acc            : {accuracy_score(y_test, pred_nb):.4f}")
    print(f"  Test Macro-F1       : {f1_score(y_test, pred_nb, average='macro'):.4f}")
    print(classification_report(y_test, pred_nb, target_names=["no", "maybe", "yes"], zero_division=0))

    axis2_results["A2c: MNB word+char"] = {
        "CV Macro-F1": f"{np.mean(fold_f1s_nb):.4f} ± {np.std(fold_f1s_nb):.4f}",
        "Test Acc": accuracy_score(y_test, pred_nb),
        "Test Macro-F1": f1_score(y_test, pred_nb, average="macro")
    }
    axis2_preds_dict["A2c_MNB_wordchar"] = pred_nb

    return (
        axis2_results,
        axis2_preds_dict,
        lr_final,
        svm_final,
        nb_final,
        word_vec_final,
        char_vec_final,
        X_cv_wc,
        X_test_wc
    )


axis2_results, axis2_preds_dict, lr_final, svm_final, nb_final, word_vec_final, char_vec_final, X_cv_wc, X_test_wc = \
    evaluate_axis2_10fold(cv_df, X_test_qctx, y_test, text_col="q_ctx")

for k, v in axis2_results.items():
    all_results[k] = v
for k, v in axis2_preds_dict.items():
    all_preds[k] = v


## Section 8 — Axis 2 Block 2: SMOTE and Soft Voting

A2d tests whether oversampling improves the minority `maybe` class. A2e combines the tuned LR, calibrated SVM, and NB models with soft voting.


In [ ]:
# SECTION 9 — BLOCK 2 — A2d and A2e

import sys
import subprocess
import numpy as np

from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import VotingClassifier

try:
    from imblearn.over_sampling import SMOTE
except ImportError:
    print("imbalanced-learn not found. Installing it now...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "imbalanced-learn"])
    from imblearn.over_sampling import SMOTE


required_vars = [
    "cv_df", "X_cv_wc", "X_test_wc", "y_cv", "y_test",
    "RANDOM_SEED", "build_word_char_features",
    "axis2_results", "axis2_preds_dict", "all_results", "all_preds",
    "svm_final", "lr_final", "nb_final"
]

missing_vars = [v for v in required_vars if v not in globals()]
if missing_vars:
    raise NameError(
        "These required variables are missing. Run the earlier cells first: "
        + ", ".join(missing_vars)
    )


# A2d: SMOTE + Logistic Regression
print("\nA2d — SMOTE + Logistic Regression (leakage-free, inside 10-fold CV)")

fold_f1s_smote = []

for fold_idx in range(10):
    val_mask = cv_df["fold"] == fold_idx
    train_mask = ~val_mask

    X_tr_texts = cv_df.loc[train_mask, "q_ctx"].tolist()
    y_tr = cv_df.loc[train_mask, "label_id"].values

    X_va_texts = cv_df.loc[val_mask, "q_ctx"].tolist()
    y_va = cv_df.loc[val_mask, "label_id"].values

    X_tr_wc, [X_va_wc], _, _ = build_word_char_features(
        X_tr_texts,
        [X_va_texts]
    )

    class_counts = np.bincount(y_tr)
    min_class_count = class_counts[class_counts > 0].min()
    smote_k = min(5, min_class_count - 1)

    if smote_k < 1:
        print(f"  Fold {fold_idx}: SMOTE skipped because a class has too few samples.")
        X_tr_sm, y_tr_sm = X_tr_wc, y_tr
    else:
        smote = SMOTE(
            random_state=RANDOM_SEED,
            k_neighbors=smote_k
        )
        X_tr_sm, y_tr_sm = smote.fit_resample(X_tr_wc, y_tr)

    clf_sm = LogisticRegression(
        max_iter=4000,
        solver="liblinear",
        class_weight="balanced",
        random_state=RANDOM_SEED,
        C=1.0
    )

    clf_sm.fit(X_tr_sm, y_tr_sm)
    val_preds = clf_sm.predict(X_va_wc)

    fold_macro_f1 = f1_score(
        y_va,
        val_preds,
        average="macro",
        zero_division=0
    )

    fold_f1s_smote.append(fold_macro_f1)

class_counts_cv = np.bincount(y_cv)
min_class_count_cv = class_counts_cv[class_counts_cv > 0].min()
smote_k_final = min(5, min_class_count_cv - 1)

if smote_k_final < 1:
    print("Final SMOTE skipped because a class has too few samples.")
    X_cv_sm, y_cv_sm = X_cv_wc, y_cv
else:
    smote_final = SMOTE(
        random_state=RANDOM_SEED,
        k_neighbors=smote_k_final
    )
    X_cv_sm, y_cv_sm = smote_final.fit_resample(X_cv_wc, y_cv)

lr_smote_final = LogisticRegression(
    max_iter=4000,
    solver="liblinear",
    class_weight="balanced",
    random_state=RANDOM_SEED,
    C=1.0
)

lr_smote_final.fit(X_cv_sm, y_cv_sm)
pred_smote = lr_smote_final.predict(X_test_wc)

smote_cv_mean = np.mean(fold_f1s_smote)
smote_cv_std = np.std(fold_f1s_smote)
smote_acc = accuracy_score(y_test, pred_smote)

smote_f1 = f1_score(
    y_test,
    pred_smote,
    average="macro",
    zero_division=0
)

print(f"  10-Fold CV Macro-F1 : {smote_cv_mean:.4f} ± {smote_cv_std:.4f}")
print(f"  Test Acc            : {smote_acc:.4f}")
print(f"  Test Macro-F1       : {smote_f1:.4f}")

print(
    classification_report(
        y_test,
        pred_smote,
        target_names=["no", "maybe", "yes"],
        zero_division=0
    )
)

axis2_results["A2d: SMOTE+LR w+c"] = {
    "CV Macro-F1": f"{smote_cv_mean:.4f} ± {smote_cv_std:.4f}",
    "Test Acc": smote_acc,
    "Test Macro-F1": smote_f1
}

axis2_preds_dict["A2d_SMOTE_LR"] = pred_smote
all_results["A2d: SMOTE+LR w+c"] = axis2_results["A2d: SMOTE+LR w+c"]
all_preds["A2d_SMOTE_LR"] = pred_smote


# A2e: Soft Voting Ensemble (LR + Cal-SVM + NB)
print("\nA2e — Soft Voting Ensemble (LR + Cal-SVM + NB)")

svm_for_ens = CalibratedClassifierCV(
    LinearSVC(
        C=svm_final.best_params_["C"],
        class_weight="balanced",
        random_state=RANDOM_SEED,
        max_iter=10000
    ),
    method="sigmoid",
    cv=3
)

lr_for_ens = LogisticRegression(
    C=lr_final.best_params_["C"],
    max_iter=4000,
    class_weight="balanced",
    solver="liblinear",
    random_state=RANDOM_SEED
)

nb_for_ens = MultinomialNB(
    alpha=nb_final.best_params_["alpha"]
)

ensemble = VotingClassifier(
    estimators=[
        ("lr", lr_for_ens),
        ("svm", svm_for_ens),
        ("nb", nb_for_ens)
    ],
    voting="soft",
    n_jobs=-1
)

ensemble.fit(X_cv_wc, y_cv)
pred_ensemble = ensemble.predict(X_test_wc)

ens_acc = accuracy_score(y_test, pred_ensemble)

ens_f1 = f1_score(
    y_test,
    pred_ensemble,
    average="macro",
    zero_division=0
)

print(f"  Test Acc            : {ens_acc:.4f}")
print(f"  Test Macro-F1       : {ens_f1:.4f}")

print(
    classification_report(
        y_test,
        pred_ensemble,
        target_names=["no", "maybe", "yes"],
        zero_division=0
    )
)

axis2_results["A2e: Soft Voting Ensemble"] = {
    "CV Macro-F1": np.nan,
    "Test Acc": ens_acc,
    "Test Macro-F1": ens_f1
}

axis2_preds_dict["A2e_Ensemble"] = pred_ensemble
all_results["A2e: Soft Voting Ensemble"] = axis2_results["A2e: Soft Voting Ensemble"]
all_preds["A2e_Ensemble"] = pred_ensemble


## Section 9 — Classical Axis 2 Summary

This table and the two plots reproduce the classical Axis 2 comparison: CV vs held-out test Macro-F1 and the confusion matrix for the best classical model selected by CV.


In [ ]:
# SECTION 9 — BLOCK 3 — SUMMARY FOR A2a–A2e

axis2_order = [
    "A2a: LR word+char",
    "A2b: Cal-SVM word+char",
    "A2c: MNB word+char",
    "A2d: SMOTE+LR w+c",
    "A2e: Soft Voting Ensemble"
]

axis2_df = pd.DataFrame.from_dict(
    {k: axis2_results[k] for k in axis2_order},
    orient="index"
)
display(axis2_df)

def get_axis2_cv_mean(name):
    cv_val = axis2_results[name]["CV Macro-F1"]
    if isinstance(cv_val, str):
        return float(cv_val.split(" ± ")[0])
    if pd.isna(cv_val):
        return -np.inf
    return float(cv_val)

best_axis2_name = max(axis2_order, key=get_axis2_cv_mean)
print(f"\nBest Axis 2 classical model (chosen by CV, not test): {best_axis2_name}")

axis2_names_list = axis2_order
axis2_cv_vals = []
axis2_test_vals = []

for k in axis2_names_list:
    cv_val = axis2_results[k]["CV Macro-F1"]
    if isinstance(cv_val, str):
        cv_mean = float(cv_val.split(" ± ")[0])
    elif pd.isna(cv_val):
        cv_mean = np.nan
    else:
        cv_mean = float(cv_val)

    axis2_cv_vals.append(cv_mean)
    axis2_test_vals.append(axis2_results[k]["Test Macro-F1"])

short_names = [n.split(": ")[1] if ": " in n else n for n in axis2_names_list]
x = np.arange(len(short_names))
w = 0.35

plt.figure(figsize=(10, 5))
plt.bar(x, axis2_cv_vals, w, label="CV Macro-F1")
plt.bar(x + w, axis2_test_vals, w, label="Test Macro-F1")
plt.xticks(x + w / 2, short_names, rotation=15)
plt.title("Axis 2 (Classical Models): CV vs Test Macro-F1")
plt.ylabel("Macro-F1")
plt.ylim(0, np.nanmax(axis2_test_vals + axis2_cv_vals) + 0.1)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(save_dir, "visualization_9_axis2_cv_vs_test_macro_f1.png"), dpi=300, bbox_inches="tight")
plt.show()

axis2_name_to_pred_key = {
    "A2a: LR word+char": "A2a_LR_wordchar",
    "A2b: Cal-SVM word+char": "A2b_CalSVM_wordchar",
    "A2c: MNB word+char": "A2c_MNB_wordchar",
    "A2d: SMOTE+LR w+c": "A2d_SMOTE_LR",
    "A2e: Soft Voting Ensemble": "A2e_Ensemble"
}

best_a2_key = axis2_name_to_pred_key[best_axis2_name]
best_a2_preds = axis2_preds_dict[best_a2_key]

cm = confusion_matrix(y_test, best_a2_preds)
plt.figure(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=["no", "maybe", "yes"],
    yticklabels=["no", "maybe", "yes"],
    cmap="Blues"
)
plt.title(f"Best Axis 2 Classical Confusion Matrix: {best_a2_key}")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.savefig(os.path.join(save_dir, "visualization_10_best_axis2_confusion_matrix.png"), dpi=300, bbox_inches="tight")
plt.show()
